In [1]:
!pip install biopython

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 42.3 MB/s eta 0:00:00


In [2]:
!pip install parasail

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.6/15.6 MB 24.9 MB/s eta 0:00:00


In [15]:
from Bio import SeqIO
import random

input_file = "/content/drive/MyDrive/bnf_new_project/sequences.fasta"
final_output = "/content/drive/MyDrive/bnf_new_project/covdata.fasta"

valid_bases = {'A', 'C', 'G', 'T'}

clean_unique = {}
removed_N = 0
removed_nonACGT = 0
duplicates = 0

for record in SeqIO.parse(input_file, "fasta"):
    seq_str = str(record.seq).upper()

    # remove sequences containing N
    if "N" in seq_str:
        removed_N += 1
        continue

    # remove sequences containing any non A,C,G,T character
    if not set(seq_str).issubset(valid_bases):
        removed_nonACGT += 1
        continue

    # deduplicate
    if seq_str not in clean_unique:
        clean_unique[seq_str] = record
    else:
        duplicates += 1


# convert dictionary to list
all_sequences = list(clean_unique.values())

if len(all_sequences) < 1000:
    raise ValueError(f"Not enough clean unique sequences ({len(all_sequences)}) to sample 1000.")

# get random subset
ref_seq = all_sequences[0]
other_seq = random.sample(all_sequences[1:], 1000)
final_set = [ref_seq] + other_seq

# save subset
SeqIO.write(final_set, final_output, "fasta")

# summary
print(f"Total original sequences: {len(all_sequences) + removed_N + removed_nonACGT + duplicates}")
print(f"Removed due to N: {removed_N}")
print(f"Removed due to non-ACGT chars: {removed_nonACGT}")
print(f"Duplicates removed: {duplicates}")
print(f"Final set contains: {len(final_set)}")
print(f"Final sample written to: {final_output}")


Total original sequences: 4852
Removed due to N: 0
Removed due to non-ACGT chars: 289
Duplicates removed: 1251
Final set contains: 1001
Final sample written to: /content/drive/MyDrive/bnf_new_project/covdata.fasta


In [17]:
print(ref_seq.id)

MT324062.1


In [ ]:
# call_snps_indels_parasail_traceback_unsplit.py
import os, gzip
from collections import namedtuple
import pandas as pd
from Bio import SeqIO
import parasail

# ---------- config ----------
INPUT_FASTA = "/content/drive/MyDrive/bnf_new_project/covdata.fasta" 
OUT_TSV     = "/content/drive/MyDrive/bnf_new_project/final_variants_snps_indels.tsv"
ALLOW_N     = False                   # set True to allow Ns and still align
# Scoring (global NW, affine gaps)
MATCH       = 2
MISMATCH    = -1
GAP_OPEN    = 5
GAP_EXTEND  = 1
# -----------------------------

Var = namedtuple("Var", "sample vtype pos ref alt length seq")

def read_genomes_from_fasta(fasta_path):
    """
    Read all genomes from a single FASTA file.
    Returns list of (id, seq_upper) tuples.
    """
    genomes = []
    opener = gzip.open if fasta_path.endswith(".gz") else open

    with opener(fasta_path, "rt") as fh:
        for rec in SeqIO.parse(fh, "fasta"):
            genomes.append((rec.id, str(rec.seq).upper()))

    if not genomes:
        raise ValueError(f"No sequences in {fasta_path}")

    return genomes

def call_variants_parasail_trace(ref_id, ref_seq, q_id, q_seq):
    """
    Global alignment (Needleman–Wunsch, affine gaps) with parasail; parse the
    traceback aligned strings (ref/query) to call SNP/INS/DEL robustly.
    Positions are 1-based reference coordinates.
    """
    mat = parasail.matrix_create("ACGT", MATCH, MISMATCH)
    res = parasail.nw_trace_scan_16(ref_seq, q_seq, GAP_OPEN, GAP_EXTEND, mat)
    tb = res.traceback
    a_ref = tb.ref   # aligned reference string (with '-')
    a_que = tb.query # aligned query string (with '-')
    L = len(a_ref)
    assert L == len(a_que)

    variants = []
    i = 0
    ref_pos = 0  # counts non-gap positions in a_ref (reference coordinate)

    while i < L:
        r = a_ref[i]
        q = a_que[i]

        # Case 1: both bases present (match/mismatch)
        if r != '-' and q != '-':
            ref_pos += 1  # we consume one reference position
            if r != q:
                variants.append(Var(q_id, "SNP", ref_pos, r, q, 1, ""))
            i += 1
            continue

        # Case 2: deletion in query relative to reference (ref has bases, query has '-')
        if r != '-' and q == '-':
            start_pos = ref_pos + 1
            del_seq = []
            # consume the full deletion run
            while i < L and a_ref[i] != '-' and a_que[i] == '-':
                del_seq.append(a_ref[i])
                ref_pos += 1
                i += 1
            seq = ''.join(del_seq)
            variants.append(Var(q_id, "DEL", start_pos, seq, f"-{len(seq)}", len(seq), seq))
            continue

        # Case 3: insertion in query relative to reference (ref '-', query bases)
        if r == '-' and q != '-':
            ins_seq = []
            # insertion occurs between ref_pos and ref_pos+1; report at pos = ref_pos+1
            while i < L and a_ref[i] == '-' and a_que[i] != '-':
                ins_seq.append(a_que[i])
                i += 1
            seq = ''.join(ins_seq)
            variants.append(Var(q_id, "INS", ref_pos + 1, "", f"+{len(seq)}", len(seq), seq))
            continue

        # Case 4: both '-' (shouldn't happen often in global NW, but just advance)
        i += 1

    return variants

def main():
    # Load all genomes from the single FASTA file
    genomes = read_genomes_from_fasta(INPUT_FASTA)

    if len(genomes) < 2:
        raise SystemExit(f"Need at least 2 genomes in {INPUT_FASTA}, found {len(genomes)}")

    # First genome is the reference
    ref_id, ref_seq = genomes[0]
    print(f"Loaded reference {ref_id} (len={len(ref_seq)})")
    print(f"Total genomes in file: {len(genomes)}")

    # Process remaining genomes as queries
    rows = []
    processed = 0
    skipped = 0

    for q_id, q_seq in genomes[1:]:
        processed += 1

        if processed % 100 == 0:
            print(f"Processed {processed}/{len(genomes)-1} genomes...")

        if q_id == ref_id:
            print(f"[skip duplicate ref] {q_id}")
            skipped += 1
            continue

        if not ALLOW_N and ("N" in q_seq or "n" in q_seq):
            print(f"[skip N] {q_id}")
            skipped += 1
            continue

        # # Check if query sequence has reasonable length
        # if len(q_seq) < len(ref_seq) * 0.5 or len(q_seq) > len(ref_seq) * 1.5:
        #     print(f"[skip length mismatch] {q_id}: ref={len(ref_seq)}, query={len(q_seq)}")
        #     skipped += 1
        #     continue

        try:
            V = call_variants_parasail_trace(ref_id, ref_seq, q_id, q_seq)

            for v in V:
                rows.append({
                    "sample": v.sample,
                    "type": v.vtype,            # SNP / INS / DEL
                    "pos": v.pos,               # 1-based ref coordinate
                    "ref": v.ref,               # ref base(s) for SNP/DEL (empty for INS)
                    "alt": v.alt,               # alt base for SNP, +len for INS, -len for DEL
                    "len": v.length,            # 1 for SNP, indel length otherwise
                    "seq": v.seq                # inserted/deleted sequence for indels; "" for SNP
                })
        except Exception as e:
            print(f"[skip error] {q_id}: {e}")
            skipped += 1
            continue

    print(f"\nProcessing complete:")
    print(f"  Reference: {ref_id}")
    print(f"  Query genomes processed: {processed - skipped}")
    print(f"  Query genomes skipped: {skipped}")

    if rows:
        df = pd.DataFrame(rows).sort_values(["pos","type","sample"])

        # Add some summary statistics
        unique_samples = df["sample"].nunique()
        variant_counts = df["type"].value_counts()

        print(f"\nVariant summary:")
        print(f"  Total variants: {len(df)}")
        print(f"  Unique samples with variants: {unique_samples}")
        for vtype, count in variant_counts.items():
            print(f"  {vtype}: {count}")

        df.to_csv(OUT_TSV, index=False)
        print(f"\nSaved variants to: {OUT_TSV}")

        # Optional: save a summary file
        summary_file = OUT_TSV.replace(".tsv", "_summary.txt")
        with open(summary_file, "w") as f:
            f.write(f"Reference genome: {ref_id}\n")
            f.write(f"Reference length: {len(ref_seq)}\n")
            f.write(f"Total genomes processed: {len(genomes)}\n")
            f.write(f"Reference genomes: 1\n")
            f.write(f"Query genomes: {len(genomes)-1}\n")
            f.write(f"Successfully processed: {processed - skipped}\n")
            f.write(f"Skipped: {skipped}\n")
            f.write(f"Total variants: {len(df)}\n")
            f.write(f"Unique samples with variants: {unique_samples}\n")
            for vtype, count in variant_counts.items():
                f.write(f"{vtype}: {count}\n")
        print(f"Summary saved to: {summary_file}")
    else:
        print("No variants found / nothing written.")

if __name__ == "__main__":
    main()

Loaded reference MT324062.1 (len=29903)
Total genomes in file: 1001
Processed 100/1000 genomes...
Processed 200/1000 genomes...
Processed 300/1000 genomes...
Processed 400/1000 genomes...
Processed 500/1000 genomes...
Processed 600/1000 genomes...
Processed 700/1000 genomes...
Processed 800/1000 genomes...
Processed 900/1000 genomes...
Processed 1000/1000 genomes...

Processing complete:
  Reference: MT324062.1
  Query genomes processed: 1000
  Query genomes skipped: 0

Variant summary:
  Total variants: 393
  Unique samples with variants: 393
  SNP: 393

Saved variants to: /content/drive/MyDrive/bnf_new_project/final_variants_snps_indels.tsv
Summary saved to: /content/drive/MyDrive/bnf_new_project/final_variants_snps_indels_summary.txt


In [20]:
# analyze mutation frequency
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# load variants
df = pd.read_csv("/content/drive/MyDrive/bnf_new_project/final_variants_snps_indels.tsv")

print(f"Total variants: {len(df)}")
print(f"Number of sequences containing variants: {df['sample'].nunique()}")
print("\nVariant type distribution:")
print(df['type'].value_counts())

Total variants: 393
Number of sequences containing variants: 393

Variant type distribution:
type
SNP    393
Name: count, dtype: int64


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from sklearn.linear_model import LinearRegression

# load data
df = pd.read_csv('/content/drive/MyDrive/bnf_new_project/final_variants_snps_indels.tsv')
df.columns = df.columns.str.strip()

BASE_DIR_1 = "/content/drive/MyDrive/bnf_new_project/stats"
BASE_DIR_2 = "/content/drive/MyDrive/bnf_new_project/plots"

# frequency analysis

type_freq = df['type'].value_counts()
pos_freq = df['pos'].value_counts()

# SNP mutations
snps = df[df['type'] == 'SNP'].copy()
snps['mutation'] = snps['ref'] + '->' + snps['alt']
mutation_freq = snps['mutation'].value_counts()

print("Mutation Type Frequencies")
print(type_freq)
print(f"\nPercentages:\n{(type_freq/len(df)*100).round(2)}")

### statistical tests

# 1. chi-square test  for mutation types
observed = type_freq.values
expected = [len(df)/len(observed)] * len(observed)
chi2, p_value = stats.chisquare(observed, expected)

print(f"\nChi-Square Test")
print(f"Chi-square: {chi2:.4f}, P-value: {p_value:.4e}")
print("Result:", "Non-uniform distribution" if p_value < 0.05 else "Uniform")

# 2. transition/traversion ratio
transitions = ['A->G', 'G->A', 'C->T', 'T->C']
ti = mutation_freq[mutation_freq.index.isin(transitions)].sum()
tv = mutation_freq[~mutation_freq.index.isin(transitions)].sum()
ti_tv_ratio = ti / tv if tv > 0 else 0

print(f"\nTi/Tv Ratio")
print(f"Transitions: {ti}, Transversions: {tv}")
print(f"Ti/Tv Ratio: {ti_tv_ratio:.3f}")

# 3. linear regression for position trend
X = pos_freq.index.values.reshape(-1, 1)
y = pos_freq.values
model = LinearRegression().fit(X, y)
r2 = model.score(X, y)

print(f"\nLinear Regression (Position vs Count)")
print(f"R²: {r2:.4f}")
print(f"Slope: {model.coef_[0]:.6f}")

# export

type_freq.to_csv(os.path.join(BASE_DIR_1, "mutation_frequencies.csv"), header=['Count'])
mutation_freq.to_csv(os.path.join(BASE_DIR_1, "snp_frequencies.csv"), header=['Count'])

stats_df = pd.DataFrame({
    'Test': ['Chi-Square', 'Ti/Tv Ratio', 'Linear R²'],
    'Value': [chi2, ti_tv_ratio, r2],
    'P-value': [p_value, np.nan, np.nan]
})
stats_df.to_csv(os.path.join(BASE_DIR_1, "statistical_results.csv"), index=False)

### plots

# 1. bar Chart for mutation types
plt.figure(figsize=(8, 5))
type_freq.plot(kind='bar', color=['#3b82f6', '#ef4444', '#10b981'])
plt.title(f'Mutation Types (p={p_value:.2e})', fontweight='bold')
plt.ylabel('Count')
plt.xticks(rotation=0)
plt.tight_layout()

plt.savefig(os.path.join(BASE_DIR_2, "mutation_types.png"), dpi=300)
plt.close()

# 2. manhattan plot
plt.figure(figsize=(12, 5))
plt.scatter(pos_freq.index, pos_freq.values, alpha=0.6)
plt.axhline(y=pos_freq.median(), color='r', linestyle='--', label='Median')
plt.title('Manhattan Plot: Mutations by Position', fontweight='bold')
plt.xlabel('Position')
plt.ylabel('Count')
plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(BASE_DIR_2, "manhattan_plot.png"), dpi=300)
plt.close()

# 3. top snp mutations
plt.figure(figsize=(10, 5))
mutation_freq.head(10).plot(kind='bar', color='#8b5cf6')
plt.title(f'Top 10 SNP Mutations (Ti/Tv={ti_tv_ratio:.2f})', fontweight='bold')
plt.ylabel('Count')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig(os.path.join(BASE_DIR_2, "top_snp_mutations.png"), dpi=300)
plt.close()

print("3 CSVs and 3 plots generated")

Mutation Type Frequencies
type
SNP    393
Name: count, dtype: int64

Percentages:
type
SNP    100.0
Name: count, dtype: float64

Chi-Square Test
Chi-square: 0.0000, P-value: nan
Result: Uniform

Ti/Tv Ratio
Transitions: 40, Transversions: 353
Ti/Tv Ratio: 0.113

Linear Regression (Position vs Count)
R²: nan
Slope: 0.000000


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_regression.py:1266: UndefinedMetricWarning: R^2 score is not well-defined with less than two samples.
  warnings.warn(msg, UndefinedMetricWarning)


3 CSVs and 3 plots generated


In [ ]:
# count how many genomes have each alternate allele at position 1
pos1 = df[df["pos"] == 1]
allele_counts = pos1["alt"].value_counts()
total = pos1["sample"].nunique()

print("Allele spectrum at position 1:")
print(allele_counts)
print(f"\nTotal genomes: {total}")
print(f"Percentages:\n{(allele_counts/total*100).round(2)}")

Allele spectrum at position 1:
alt
A    393
Name: count, dtype: int64

Total genomes: 393
Percentages:
alt
A    100.0
Name: count, dtype: float64


In [27]:
BASE_DIR = "/content/drive/MyDrive/bnf_new_project"

In [ ]:
# find mutation
from Bio import AlignIO
import pandas as pd
from collections import Counter
import numpy as np


ref_type = "specific"
ref_id = "MT324062.1"

#load alignment
aln = AlignIO.read("/content/drive/MyDrive/bnf_new_project/final_aligned.fas", "fasta")
seqs = [str(record.seq).upper() for record in aln]
ids = [record.id for record in aln]
seq_length = aln.get_alignment_length()

if ref_type == "consensus":
# build specific reference
    cons = ""
    for c in range(seq_length):
        bases = [s[c] for s in seqs if s[c] not in "-N"]
        cons += Counter(bases).most_common(1)[0][0] if bases else "N"
elif ref_type == "specific":
    if ref_id not in ids:
        raise ValueError(f"Reference ID '{ref_id}' not found in FASTA IDs.")
    cons = seqs[ids.index(ref_id)]
else:
    raise ValueError("ref_type must be 'consensus' or 'specific'.")

# find mutations
rows = []
for pos in range(seq_length):
    ref = cons[pos]
    for sid, s in zip(ids, seqs):
        alt = s[pos]
        if alt not in "-N" and alt != ref:
            rows.append([pos+1, sid, ref, alt, f"{ref}{pos+1}{alt}"])

mut = pd.DataFrame(rows, columns=["Position","Sample","Ref","Alt","Mutation"])

# explore
print(mut.head())          # show first few mutations
print("Total mutations:", len(mut))

mut.to_csv(os.path.join(BASE_DIR, "final_cov_mutations.csv"), index=False)
def valid(b):
    return b not in "-N"

# generate mutation distance matrix
def fast_pairwise_distance(seqs, ids):
    n = len(seqs)
    L = len(seqs[0])

    # preprocess: mark valid positions
    valid_positions = []
    for pos in range(L):
        # check if this position has any non gap, non-N in any sequence
        has_valid = False
        for s in seqs:
            if s[pos] not in '-N':
                has_valid = True
                break
        if has_valid:
            valid_positions.append(pos)

    print(f"Comparing at {len(valid_positions)} valid positions (ignoring all-gap columns)")

    D = np.zeros((n, n), dtype=int)

    for i in range(n):
        si = seqs[i]
        for j in range(i+1, n):
            sj = seqs[j]
            d = 0
            for pos in valid_positions:
                a, b = si[pos], sj[pos]
                if a not in '-N' and b not in '-N' and a != b:
                    d += 1
            D[i, j] = D[j, i] = d

        if i % 100 == 0:
            print(f"Processed {i}/{n} sequences")

    return D

print("computing")
D = fast_pairwise_distance(seqs, ids)
D_df = pd.DataFrame(D, index=ids, columns=ids)
D_df.to_csv(os.path.join(BASE_DIR, "final_distance_matrix.csv"))

   Position      Sample Ref Alt Mutation
0         1  OM721096.1   A   T      A1T
1         1  OQ352750.1   A   C      A1C
2         1  PP522198.1   A   T      A1T
3         1  MT576584.1   A   T      A1T
4         1  OQ924220.1   A   T      A1T
Total mutations: 18348298
computing
Comparing at 29903 valid positions (ignoring all-gap columns)
Processed 0/1001 sequences
Processed 100/1001 sequences
Processed 200/1001 sequences
Processed 300/1001 sequences
Processed 400/1001 sequences
Processed 500/1001 sequences
Processed 600/1001 sequences
Processed 700/1001 sequences
Processed 800/1001 sequences
Processed 900/1001 sequences
Processed 1000/1001 sequences


In [30]:
import pandas as pd
import numpy as np
from scipy.sparse.csgraph import minimum_spanning_tree

# load distance matrix
df = pd.read_csv("/content/drive/MyDrive/bnf_new_project/final_distance_matrix.csv", index_col=0)

print(f"Distance matrix shape: {df.shape}")
print(f"Sample IDs: {df.index.tolist()[:5]}...")  # showing first 5

# convert to numpy array (keeping index for reference)
distance_matrix = df.values
sample_ids = df.index.tolist()

# MST
mst_sparse = minimum_spanning_tree(distance_matrix)
mst_array = mst_sparse.toarray().astype(int)

# create DataFrame with labels
mst_df = pd.DataFrame(mst_array, index=sample_ids, columns=sample_ids)

print(f"MST shape: {mst_df.shape}")
print(f"Number of edges: {np.sum(mst_array > 0)}")  #  n_samples - 1

# save MST

mst_df.to_csv(os.path.join(BASE_DIR, "final_mst_results.csv"))
print("MST saved")

# save edge list for easier network analysis
edges = []
for i in range(len(sample_ids)):
    for j in range(i+1, len(sample_ids)):
        if mst_array[i, j] > 0:
            edges.append([sample_ids[i], sample_ids[j], mst_array[i, j]])

edges_df = pd.DataFrame(edges, columns=['Source', 'Target', 'Weight'])
edges_df.to_csv(os.path.join(BASE_DIR, "final_mst_edges.csv"), index=False)
print(f"Edge list saved: {len(edges)} edges")

Distance matrix shape: (1001, 1001)
Sample IDs: ['MT324062.1', 'OM721096.1', 'OM773328.1', 'OL351375.1', 'OL601560.1']...
MST shape: (1001, 1001)
Number of edges: 1000
MST saved
Edge list saved: 1000 edges


In [33]:
# extract accession to use for metadata look up
import re

input_fasta = "/content/drive/MyDrive/bnf_new_project/covdata.fasta"
output_file = "/content/drive/MyDrive/bnf_new_project/accessions.txt"

with open(input_fasta, "r") as infile, open(output_file, "w") as outfile:
    for line in infile:
        if line.startswith(">"):
            accession = line[1:].split()[0]
            outfile.write(accession + "\n")

In [ ]:
import pandas as pd
import numpy as np
from scipy.sparse.csgraph import minimum_spanning_tree
from collections import deque


# load distance matrix
dist = pd.read_csv("/content/drive/MyDrive/bnf_new_project/final_distance_matrix.csv", index_col=0)

ids_all = dist.index.tolist()

# load metadata and build ID (date lookup)
meta = pd.read_csv("/content/drive/MyDrive/bnf_new_project/final_metadata.csv", dtype=str)

# parse Collection_Date into datetime
meta["Collection_Date"] = pd.to_datetime(meta["Collection_Date"], errors="coerce")

# lookup dictionary
date_lookup = dict(zip(meta["Accession"], meta["Collection_Date"]))

# BFS: build parent-child map from adjacency list
def bfs_parent(graph, root):
    parent = {root: None}
    visited = {root}
    q = deque([root])

    while q:
        node = q.popleft()
        for neigh in graph[node]:
            if neigh not in visited:
                visited.add(neigh)
                parent[neigh] = node
                q.append(neigh)

    return parent


# evaluate MST success for one subsample of genomes

def evaluate_once(sample_ids):
    # extract the submatrix
    sub = dist.loc[sample_ids, sample_ids].values

    # compute MST
    mst = minimum_spanning_tree(sub).toarray()

    # build adjacency list
    adj = {sid: [] for sid in sample_ids}
    n = len(sample_ids)
    for i in range(n):
        for j in range(n):
            if mst[i, j] > 0:        # MST edge
                a = sample_ids[i]
                b = sample_ids[j]
                adj[a].append(b)
                adj[b].append(a)

    # pick root = earliest collection date
    def get_date(x):
        return date_lookup.get(x, pd.NaT)

    root = min(sample_ids, key=lambda x: get_date(x) if pd.notna(get_date(x)) else pd.Timestamp.max)

    parent_map = bfs_parent(adj, root)

    # compare parent and child
    success = 0
    total = 0

    for child, parent in parent_map.items():
        if parent is None:
            continue  # skip root

        d_parent = get_date(parent)
        d_child = get_date(child)

        if pd.isna(d_parent) or pd.isna(d_child):
            continue  # skip if missing date

        total += 1
        if d_parent <= d_child:
            success += 1

    if total == 0:
        return np.nan
    return success / total

# run 1000 iterations

success_rates = []
N_ITER = 1000
SAMPLE_SIZE = min(1000, len(ids_all))

for i in range(N_ITER):
    subset = np.random.choice(ids_all, SAMPLE_SIZE, replace=False)
    rate = evaluate_once(subset)
    success_rates.append(rate)

    if (i+1) % 50 == 0:
        print(f"Iteration {i+1}/{N_ITER}  success_rate={rate:.4f}")

print("\nAverage success rate:", np.nanmean(success_rates))

### save results

# save per iteration success rates
results = pd.DataFrame({
    "iteration": np.arange(1, N_ITER+1),
    "success_rate": success_rates
})

results.to_csv(os.path.join(BASE_DIR, "success_rates.csv"), index=False)
print("Saved: success_rates.csv")

# save summary statistics
summary = {
    "mean_success_rate": float(np.nanmean(success_rates)),
    "median_success_rate": float(np.nanmedian(success_rates)),
    "std_success_rate": float(np.nanstd(success_rates)),
    "min_success_rate": float(np.nanmin(success_rates)),
    "max_success_rate": float(np.nanmax(success_rates)),
    "iterations": N_ITER
}

summary_df = pd.DataFrame([summary])
summary_df.to_csv(os.path.join(BASE_DIR, "success_summary.csv"), index=False)
print("Saved: success_summary.csv")

print("\nSummary:")
print(summary_df)

Iteration 50/1000  success_rate=0.6091
Iteration 100/1000  success_rate=0.6138
Iteration 150/1000  success_rate=0.6131
Iteration 200/1000  success_rate=0.6077
Iteration 250/1000  success_rate=0.6229
Iteration 300/1000  success_rate=0.6087
Iteration 350/1000  success_rate=0.6131
Iteration 400/1000  success_rate=0.6229
Iteration 450/1000  success_rate=0.6121
Iteration 500/1000  success_rate=0.6061
Iteration 550/1000  success_rate=0.6117
Iteration 600/1000  success_rate=0.6081
Iteration 650/1000  success_rate=0.6242
Iteration 700/1000  success_rate=0.6188
Iteration 750/1000  success_rate=0.5980
Iteration 800/1000  success_rate=0.6242
Iteration 850/1000  success_rate=0.6192
Iteration 900/1000  success_rate=0.6299
Iteration 950/1000  success_rate=0.6097
Iteration 1000/1000  success_rate=0.6097

Average success rate: 0.6123358531041794
Saved: success_rates.csv
Saved: success_summary.csv

Summary:
   mean_success_rate  median_success_rate  std_success_rate  min_success_rate  \
0           0.6

In [ ]:
import pandas as pd

# define get_date function globally
def get_date(x):
    return date_lookup.get(x, pd.NaT)

# determine root using the last generated subset from the previous iteration
root = min(subset, key=lambda x: get_date(x) if pd.notna(get_date(x)) else pd.Timestamp.max)

print("Root genome:", root, "Date:", get_date(root))

Root genome: MT324062.1 Date: 2020-03-07 00:00:00
